# Retrieval-Augmented Generation (RAG)

### INFO 5940 · Assignment 2

In class, we saw how a model can answer questions using information from a
document or database. In this notebook, you will work through that process using
**Python code**. You will see how we find relevant information, include it in a
request, and use it to generate an answer.

We will start with a text document, split it into smaller pieces, and search
for passages related to a question. Then we will see how changing the retrieved
passages affects the answer. Finally, we will use SQL to retrieve information
from a database and answer questions about it.

As you work through the examples, focus on what information reaches the model
and whether that information supports its answer. Compare the results and
explain what you think happened.

## How to Use This Notebook

Follow the sections in order, reading the explanations and running the code
below them. Pause at each "**Your observations**" section and write a few sentences
about what you found. Keep your outputs visible so you can refer back to them.

| Section | What you will learn |
| --- | --- |
| A. Set up the model | Prepare the model interface for our requests |
| B. Load the document | Read the source text and inspect its metadata |
| C. Split the document | See how chunk boundaries affect context |
| D. Index the chunks in Chroma | Create embeddings and store the chunks |
| E. Inspect the search results | Compare retrieved passages and their distances |
| F. Prepare the retrieved context | Choose how many chunks to retrieve and format their text |
| G. Build the prompt and RAG chain | Connect retrieval, the prompt, and answer generation |
| H. Retrieve fewer chunks | Compare answers using different amounts of context |
| I. Use a model to help choose chunk boundaries | Keep related passages together |
| J. Search the new chunks | Compare retrieval after changing the chunks |
| K. Open and inspect the database | Explore the Chinook tables and run a SQL query |
| L. Generate SQL and inspect the prompt | See how the model uses the database schema |
| M. Clean and execute the SQL | Run the generated query and inspect its result |
| N. Connect SQL generation and execution | Combine the steps into a chain |
| O. Turn the result into an answer | Give database results to the answering model |
| P. Consider the limits of the evidence | Check whether the records support the conclusion |

Sections A through J cover document RAG. Sections K through P cover structured database RAG.

## Part 1: RAG with a text document

### A. Set up the model

We will use `ChatOpenAI`, LangChain's interface for making chat model calls.
The next two cells import it and create `llm` with the model settings. Creating
this object prepares us to make requests; it does not call the model yet.

In [1]:
from langchain_openai import ChatOpenAI
from os import environ

In [2]:
llm = ChatOpenAI(
    model="openai.gpt-4o",
    temperature=0.2,
)

### B. Load the document

`TextLoader` reads the file into a list of Document objects. `documents[0]`
selects the first one, as the documents are zero-indexed. Its `page_content` holds the text, and its `metadata`
records information such as the source filename. This is how a retrieval system
can keep track of where a passage came from.

Read the `RAG_source.txt` document once before moving on. In particular, locate sections that relate to the drug's use during 
pregnancy. Please note that the company
and drug scenarios are fictional classroom examples, not medical guidance.

In [3]:
from langchain_community.document_loaders import TextLoader
from pathlib import Path

# Supports running from the rag_notebooks folder or the repository root.
data_path = Path("data/RAG_source.txt")
if not data_path.exists():
    data_path = Path("rag_notebooks/data/RAG_source.txt")
loader = TextLoader(str(data_path), encoding="utf-8")
documents = loader.load()

In [4]:
documents[0].metadata

{'source': 'data/RAG_source.txt'}

In [5]:
print(documents[0].page_content)

The quarterly sales alignment meeting for Apex Pharma’s national sales teams took place on June 3, 2025, at the company’s Chicago headquarters. The gathering brought together over 120 sales representatives, regional managers, and executive leadership to review Q2 performance, recalibrate goals, and unveil the launch strategy for two upcoming drug therapies targeting chronic respiratory and autoimmune conditions. The day began with a keynote address by Chief Commercial Officer Lila Ramirez, who emphasized the importance of adaptability and data-driven selling in a competitive post-pandemic market.

Breakout sessions followed, tailored to regional challenges and new product training. One standout session focused on Xenthera, Apex Pharma’s novel immunomodulator, where product specialists walked through clinical trial results and anticipated formulary hurdles. Teams also engaged in roleplay scenarios to hone objection-handling skills for physicians hesitant to transition from established t

### C. Split the document

The file is small enough to read in full, but we will split it to see how the
same process works with a larger collection, since a model's context window is a known limitation. A chunk is a piece of the document
that we can retrieve separately.

Here, `chunk_size=200` counts **characters**, not tokens. The recursive splitter
tries paragraph boundaries, then smaller boundaries such as line breaks and
spaces. `chunk_overlap=0` requests no shared text between adjacent chunks.
Creating `text_splitter` sets the rules; `split_documents` applies them.

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [7]:
chunk_size = 200
chunk_overlap = 0

In [8]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = chunk_size,
    chunk_overlap = chunk_overlap
)

In [9]:
chunks = text_splitter.split_documents(documents)

In [10]:
for chunk in chunks:
    print(chunk.page_content)
    print("-----")

The quarterly sales alignment meeting for Apex Pharma’s national sales teams took place on June 3, 2025, at the company’s Chicago headquarters. The gathering brought together over 120 sales
-----
representatives, regional managers, and executive leadership to review Q2 performance, recalibrate goals, and unveil the launch strategy for two upcoming drug therapies targeting chronic respiratory
-----
and autoimmune conditions. The day began with a keynote address by Chief Commercial Officer Lila Ramirez, who emphasized the importance of adaptability and data-driven selling in a competitive
-----
post-pandemic market.
-----
Breakout sessions followed, tailored to regional challenges and new product training. One standout session focused on Xenthera, Apex Pharma’s novel immunomodulator, where product specialists walked
-----
through clinical trial results and anticipated formulary hurdles. Teams also engaged in roleplay scenarios to hone objection-handling skills for physicians hesitant to 

### Your observations 1

Choose one place where the splitter separates text that you would want to keep
together. Quote a short phrase and explain where you would put the boundary.
What is one downside of simply making every chunk much larger?

**Your response:**

The splitter cut the toxicology file in the middle of a sentence. One chunk ends with "showed a" and the next chunk starts with "29 % decrease in offspring viability". That second chunk then ends with "administration is contraindicated for", and the days of pregnancy are left alone in a third chunk. I would put the boundary before "Restricted Toxicology File" and after the sentence about the contraindicated days, so the study, the result and the classification stay in one chunk. The downside of simply making every chunk much larger is that one chunk would mix different topics, for example the sales meeting and the Zelomax leaflet. Then the chunk is less specific for the search, and the model also receives more text that is not related to the question.

### D. Index the chunks in Chroma

An embedding represents text as a list of numbers. One chunk gets one vector;
the vector length depends on the embedding model and its settings, rather than
how many words the chunk contains.

`Chroma.from_documents` uses the embedding model to embed our chunks, then stores
the text, vectors, and metadata in a vector database. The embedding model helps
us find related text. The chat model will write the answer later.

In [11]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

In [12]:
vectorstore = Chroma.from_documents(documents=chunks, embedding=OpenAIEmbeddings(model="openai.text-embedding-3-large"))

### E. Inspect the search results

The search embeds the question using the same embedding model and finds nearby
chunk vectors. It returns stored passages, not an answer generated by the chat
model. The second cell also returns distances: **lower means closer** for this
method. A close match does not necessarily contain the most important evidence.

Recall the class example asking where Ithaca is: the retriever still returned
passages from this document. It selects nearby vectors even when none of the
passages answers the question. Retrieval and answer generation are separate steps.

In [13]:
vectorstore.similarity_search("Is Zelomax safe during pregnancy?")

[Document(metadata={'source': 'data/RAG_source.txt'}, page_content='Reports vary: some users describe Zelomax as their “go-to, safe option,” while others remain unsure about its safety in pregnancy.'),
 Document(metadata={'source': 'data/RAG_source.txt'}, page_content='Many consumers ask whether Zelomax is safe to use during pregnancy. Online forums feature dozens of posts comparing the perceived safety of Zelomax to older antihistamines, with users debating how'),
 Document(metadata={'source': 'data/RAG_source.txt'}, page_content='Clinical newsletters routinely review the safety profile of Zelomax. Several case summaries note that, when prescribed during pregnancy, clinicians carefully weigh perceived safety benefits against'),
 Document(metadata={'source': 'data/RAG_source.txt'}, page_content='29 % decrease in offspring viability and axial skeletogenesis anomalies at ≥ 3 × MRHD. Zelomax™ has therefore been classified as very dangerous during pregnancy; administration is contraindicat

In [14]:
vectorstore.similarity_search_with_score("Is Zelomax safe during pregnancy?")

[(Document(metadata={'source': 'data/RAG_source.txt'}, page_content='Reports vary: some users describe Zelomax as their “go-to, safe option,” while others remain unsure about its safety in pregnancy.'),
  0.40547090768814087),
 (Document(metadata={'source': 'data/RAG_source.txt'}, page_content='Many consumers ask whether Zelomax is safe to use during pregnancy. Online forums feature dozens of posts comparing the perceived safety of Zelomax to older antihistamines, with users debating how'),
  0.41786015033721924),
 (Document(metadata={'source': 'data/RAG_source.txt'}, page_content='Clinical newsletters routinely review the safety profile of Zelomax. Several case summaries note that, when prescribed during pregnancy, clinicians carefully weigh perceived safety benefits against'),
  0.6464887261390686),
 (Document(metadata={'source': 'data/RAG_source.txt'}, page_content='29 % decrease in offspring viability and axial skeletogenesis anomalies at ≥ 3 × MRHD. Zelomax™ has therefore been cla

### F. Prepare the retrieved context

`as_retriever` creates a step we can call with a question. Here, `k=10` asks for
ten chunks using similarity search. `format_docs` joins their `page_content`
with blank lines. It leaves out the metadata and scores.

The output below is the text we can pass to the model. If you see `\n` in the
notebook's string display, those characters represent line breaks.

In [15]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 10})

In [16]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [17]:
format_docs(retriever.invoke("Is Zelomax safe during pregnancy?"))

'Reports vary: some users describe Zelomax as their “go-to, safe option,” while others remain unsure about its safety in pregnancy.\n\nMany consumers ask whether Zelomax is safe to use during pregnancy. Online forums feature dozens of posts comparing the perceived safety of Zelomax to older antihistamines, with users debating how\n\nClinical newsletters routinely review the safety profile of Zelomax. Several case summaries note that, when prescribed during pregnancy, clinicians carefully weigh perceived safety benefits against\n\n29 % decrease in offspring viability and axial skeletogenesis anomalies at ≥ 3 × MRHD. Zelomax™ has therefore been classified as very dangerous during pregnancy; administration is contraindicated for\n\n“safe” or “not so safe” it felt for them personally in each trimester of pregnancy. Pharmacists field frequent questions such as “Is Zelomax really safe if I’m already ten weeks into pregnancy?”\n\nlimited long-term data. Advisory columns phrase it this way: “I

### G. Build the prompt and RAG chain

This is the augmentation step. `PromptTemplate` has two placeholders:
`{question}` for the user's question and `{context}` for the retrieved text.
Read the instructions, including what the model is asked to do when it does
not know the answer.

In [18]:
from langchain_core.prompts import PromptTemplate

template = """
    You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. 
    If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
    
    Question: {question} 
    
    Context: {context} 
    
    Answer:
"""
prompt = PromptTemplate.from_template(template)

The `|` operator connects steps. The first dictionary prepares the prompt values:
`retriever | format_docs` supplies the context, while `RunnablePassthrough()`
passes the question through unchanged. The prompt goes to `llm`, and
`StrOutputParser()` extracts the reply text. `.invoke(...)` runs the sequence.

In [19]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [20]:
rag_chain.invoke("Is Zelomax safe during pregnancy?")

'Zelomax is classified as very dangerous during pregnancy, with studies showing a 29% decrease in offspring viability and skeletal anomalies at higher doses. Its use is contraindicated during pregnancy. Consult your healthcare provider for safer alternatives.'

### H. Retrieve fewer chunks

Before running these cells, predict whether retrieving only two chunks will
change the answer. Put that prediction in the response area below. Then inspect
the context, rebuild the chain, and ask the same question.

Reassigning `retriever` alone does not update the retriever held inside the
earlier chain.

In [21]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 2})

In [22]:
format_docs(retriever.invoke("Is Zelomax safe during pregnancy?"))

'Reports vary: some users describe Zelomax as their “go-to, safe option,” while others remain unsure about its safety in pregnancy.\n\nMany consumers ask whether Zelomax is safe to use during pregnancy. Online forums feature dozens of posts comparing the perceived safety of Zelomax to older antihistamines, with users debating how'

In [23]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [24]:
rag_chain.invoke("Is Zelomax safe during pregnancy?")

'The safety of Zelomax during pregnancy is unclear, as opinions and reports vary. Some users consider it safe, while others remain uncertain. It is best to consult a healthcare professional for guidance.'

### Your observations 2

Record your prediction first, then compare it with the answers for `k=10` and
`k=2`. Identify a retrieved passage that helps explain what happened. What does
your result show about the difference between text that is similar to a question
and text that is needed to answer it? The answers do not have to differ.

**Your response:**

Before running, my prediction was that the answer would change with k=2, because my question uses the word safe and the consumer leaflet repeats that word many times. The result matched my prediction. With k=10 the answer said Zelomax is classified as very dangerous during pregnancy and that its use is contraindicated. With k=2 the answer said the safety is unclear because opinions and reports vary. The two passages retrieved with k=2 were "Reports vary: some users describe Zelomax" and "Many consumers ask whether Zelomax is safe to use during pregnancy". These passages are very similar to my question because they repeat Zelomax, safe and pregnancy. However, they only describe what people feel. The passage needed for the answer is the toxicology one, and it says dangerous and contraindicated and does not use the word safe at all. So it was not in the top 2 and only came in when I retrieved 10 chunks. This shows that text similar to a question and text needed to answer it are not the same thing.

### I. Use a model to help choose chunk boundaries

The first splitter followed character counts and text boundaries. The next
function asks a model whether two pieces continue the same idea. Its score
runs from 0 to 1, with higher values indicating a stronger connection. This
is a model-generated judgment, not a vector distance.

The example call lets us try the judgment on two related passages before using
it to build chunks across the document.

In [25]:
documents

[Document(metadata={'source': 'data/RAG_source.txt'}, page_content='The quarterly sales alignment meeting for Apex Pharma’s national sales teams took place on June 3, 2025, at the company’s Chicago headquarters. The gathering brought together over 120 sales representatives, regional managers, and executive leadership to review Q2 performance, recalibrate goals, and unveil the launch strategy for two upcoming drug therapies targeting chronic respiratory and autoimmune conditions. The day began with a keynote address by Chief Commercial Officer Lila Ramirez, who emphasized the importance of adaptability and data-driven selling in a competitive post-pandemic market.\n\nBreakout sessions followed, tailored to regional challenges and new product training. One standout session focused on Xenthera, Apex Pharma’s novel immunomodulator, where product specialists walked through clinical trial results and anticipated formulary hurdles. Teams also engaged in roleplay scenarios to hone objection-ha

In [26]:
def llm_similarity(text1, text2):
    template = """
    Analyze the contextual relationship between the following two texts:
    
    Text 1: {text1}
    Text 2: {text2}
    
    Evaluate whether Text 2 completes or extends the context of Text 1, or if they are separate and unrelated. Assign a float score from 0 to 1, where:
    
    0 = The texts are entirely unrelated and should be split
    1 = The texts are strongly connected and belong to the same context
    
    Consider factors such as:
    
    Thematic continuity
    Logical flow
    Shared subject matter
    Narrative or argumentative progression
    Linguistic cohesion
    Provide only a single float value between 0 and 1 as your response, with up to two decimal places. For example: 0.75
    
    Ensure your answer contains nothing but the float value. Double-check your response before submitting"""
    
    prompt = PromptTemplate.from_template(template)
    chain = prompt | llm | StrOutputParser()

    similarity = chain.invoke({"text1": text1, "text2": text2})
    return float(similarity.replace('.\n\n', ''))

In [27]:
llm_similarity("""Introducing Xenthera™ – The Next Generation in Chronic Respiratory Care""",
               """Help your patients move beyond the flare-up cycle. Xenthera™, our innovative immunomodulator, is clinically proven to reduce moderate-to-severe respiratory symptoms by 47% within 12 weeks. With
""")

0.95

`merge_metadata` combines the metadata of pieces that are joined. In this file,
the pieces have the same source path, so the helper may repeat that path.
The repetition does not mean the text came from multiple files.

In [28]:
def merge_metadata(metadatas):
    merged_metadata = {}
    for metadata in metadatas:
        for key, value in metadata.items():
            if key in merged_metadata:
                merged_metadata[key] += " " + value  
            else:
                merged_metadata[key] = value
    return merged_metadata

The splitting function has three main steps:

1. Split the source into small pieces and keep their text and metadata.
2. Start a chunk and compare it with the next piece. Merge them if the score
   is above `0.49` and the combined length passes the size check.
3. Save completed chunks, keep any remaining text, and return Documents.

The call uses 100-character pieces, starts with groups of two pieces
(`200 // 100`), and sets a 4,000-character limit for further merging. It makes
several model calls, so expect it to take longer than the first splitter.

In [29]:
from langchain_core.documents import Document

def context_text_splitter_with_llm(documents, step_size, chunk_size, max_chunk_size):
    # Split the text
    # Ensure you have RecursiveCharacterTextSplitter defined and available
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=step_size, chunk_overlap=0)

    docs = text_splitter.split_documents(documents)

    step_chunks = [doc.page_content for doc in docs]
    step_metadata = [doc.metadata for doc in docs]

    merged_chunks = []
    merged_metadata_chunks = []

    while len(step_chunks) > 0:
        if len(''.join(step_chunks)) < chunk_size:
            break
        
        current_chunk = ''.join(step_chunks[:chunk_size//step_size])
        current_metadata = merge_metadata(step_metadata[:chunk_size//step_size])
        
        step_chunks = step_chunks[chunk_size//step_size:]
        step_metadata = step_metadata[chunk_size//step_size:]

        chunk_appended = False

        while len(step_chunks) > 0:
            next_step_chunk = step_chunks.pop(0)
            next_step_chunk_metadata = step_metadata.pop(0)

            similarity_score = llm_similarity(current_chunk, next_step_chunk)

            if similarity_score > 0.49 and len(current_chunk) + len(next_step_chunk) <= max_chunk_size:
                current_chunk += " " + next_step_chunk
                current_metadata = merge_metadata([current_metadata, next_step_chunk_metadata])
            else:
                merged_chunks.append(" " + current_chunk)
                merged_metadata_chunks.append(current_metadata)
                
                chunk_appended = True
                
                step_chunks.insert(0, next_step_chunk)
                step_metadata.insert(0, next_step_chunk_metadata)
                
                break

        if not chunk_appended:
            merged_chunks.append(" " + current_chunk)
            merged_metadata_chunks.append(current_metadata)

    if len(step_chunks) > 0:
        merged_chunks.append(' '.join(step_chunks))
        merged_metadata_chunks.append(merge_metadata(step_metadata))

    merged_docs = []
    for chunk, metadata in zip(merged_chunks, merged_metadata_chunks):
        merged_docs.append(Document(page_content=chunk, metadata=metadata))

    return merged_docs

In [30]:
chunks = context_text_splitter_with_llm(documents, 100, 200, 4000)

In [31]:
for chunk in chunks:
    print(chunk.page_content)
    print("-----")

 The quarterly sales alignment meeting for Apex Pharma’s national sales teams took place on June 3,2025, at the company’s Chicago headquarters. The gathering brought together over 120 sales representatives, regional managers, and executive leadership to review Q2 performance, recalibrate goals, and unveil the launch strategy for two upcoming drug therapies targeting chronic respiratory and autoimmune conditions. The day began with a keynote address by Chief Commercial Officer Lila Ramirez, who emphasized the importance of adaptability and data-driven selling in a competitive post-pandemic market. Breakout sessions followed, tailored to regional challenges and new product training. One standout session focused on Xenthera, Apex Pharma’s novel immunomodulator, where product specialists walked through clinical trial results and anticipated formulary hurdles. Teams also engaged in roleplay scenarios to hone objection-handling skills for physicians hesitant to transition from established tr

### J. Search the new chunks

Changing `chunks` does not update the database. The next cells delete the old
collection and index the new chunks. We then retrieve one chunk and rebuild
the same RAG chain.

One new chunk can hold much more text than an earlier chunk. We have changed
both the chunk boundaries and `k`, so this comparison does not isolate either
change on its own.

In [32]:
# clear the vectorstore
vectorstore.delete_collection()

In [33]:
vectorstore = Chroma.from_documents(documents=chunks, embedding=OpenAIEmbeddings(model="openai.text-embedding-3-large"))

In [34]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 1})

In [35]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [36]:
rag_chain.invoke("Is Zelomax safe during pregnancy?")

'The safety of Zelomax during pregnancy is uncertain due to limited long-term data. Clinicians recommend discussing its use with your healthcare provider to weigh potential benefits against risks. It is best to consult your care team for personalized advice.'

### Your observations 3

Find a new chunk that keeps related information together and compare it with
the earlier split. Did that change help the answer in your run? Explain whether
the benefit you observed seems worth the additional model calls. Remember that
this example also changed `k`, so chunking alone cannot explain every difference.

**Your response:**

The new splitter gave me 4 chunks instead of 34. The third chunk keeps the whole toxicology file together: the title, the rabbit study, the 29 % decrease and the sentence that Zelomax is classified as very dangerous during pregnancy. In the earlier split this same text was broken into four pieces. However, this change did not help the answer in my run. With k=1 the answer said the safety is uncertain because of limited data and told me to talk to my care team. That wording comes from the consumer leaflet chunk, so the retriever picked the leaflet chunk and not the toxicology chunk. The new splitter also made many model calls and took much longer than the first splitter. For a small document like this one, the benefit I observed is not worth the additional calls. Also k changed from 2 to 1 at the same time, so I cannot say the chunking alone caused this result.

## Part 2: RAG with a structured database

For a question such as “How many employees are there?”, retrieving a few similar
records would not give us the total. SQL can count the records directly.
We will let a model write SQL, execute it, and include the results in an
answering prompt.

### K. Open and inspect the database

[Chinook](https://github.com/lerocha/chinook-database) is a sample music-store
database with artists, albums, tracks, customers, employees, and sales.
`SQLDatabase` lets LangChain inspect its tables and run queries. SQLite reads
from the supplied file; `mode=ro` opens it read-only.

The first query selects up to ten rows from `Artist`. pandas displays the
result as a table. No model call is needed for these steps.

In [37]:
# These helpers display formatted output in the notebook.
# We use Markdown below to render the model's answers.
from IPython.display import display, Markdown, HTML


In [38]:
# SQLDatabase lets LangChain inspect tables and run SQL queries.
# pandas displays query results as a table; Path helps locate our database file.
from langchain_community.utilities import SQLDatabase
from IPython.display import display
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine

# SQLite stores this entire sample database in one file, so no server is needed.
# The first path works from this notebook's folder; the second works from the repo root.
db_path = Path("data/Chinook.db")
if not db_path.is_file():
    db_path = Path("assignments/02-structured-data-rag/data/Chinook.db")
if not db_path.is_file():
    raise FileNotFoundError("Open this notebook from the repository root or assignments/02-structured-data-rag folder.")

# This connection address identifies SQLite and the full path to the file.
# mode=ro opens it read-only, so generated queries cannot change its records.
db_uri = f"sqlite:///file:{db_path.resolve().as_posix()}?mode=ro&uri=true"
db = SQLDatabase.from_uri(db_uri)

# The dialect tells us which variety of SQL to use. The table names tell us
# what data we can query. None of these operations calls a language model.
display(db.dialect)
display(db.get_usable_table_names())

# An engine manages database connections. pandas uses this one to execute SQL.
engine = create_engine(db_uri)

# Start with a query we wrote ourselves: show up to 10 rows from Artist.
query = "SELECT * FROM Artist LIMIT 10;"
df = pd.read_sql_query(query, engine)
display(df)


'sqlite'

['Album',
 'Artist',
 'Customer',
 'Employee',
 'Genre',
 'Invoice',
 'InvoiceLine',
 'MediaType',
 'Playlist',
 'PlaylistTrack',
 'Track']

,ArtistId,Name
0,1,AC/DC
1,2,Accept
2,3,Aerosmith
3,4,Alanis Morissette
4,5,Alice In Chains
5,6,Antônio Carlos Jobim
6,7,Apocalyptica
7,8,Audioslave
8,9,BackBeat
9,10,Billy Cobham


### L. Generate SQL and inspect the prompt

We now create the model interface for the database example. This reassigns
`llm`; the next cells also reuse `chain` and `response` as the example develops.
Those names are ordinary Python variables.

`create_sql_query_chain` sends the question, table definitions, and sample rows
to the model. It returns SQL text; it does not execute the SQL. The last cell
shows the prompt template that LangChain uses to request it.

In [39]:
# ChatOpenAI is LangChain's interface to the model.
# It reads OPENAI_API_KEY and OPENAI_BASE_URL from the Codespace environment.
from langchain_openai import ChatOpenAI
from os import environ

# This creates the model interface. A later .invoke(...) call sends the request.
llm = ChatOpenAI(model="openai.gpt-4o")


In [40]:
from langchain.chains import create_sql_query_chain

# A chain connects processing steps. This one sends the question, table
# definitions, and sample rows to the model so it can write SQLite SQL.
# It returns SQL text; it does not execute the generated query.
chain = create_sql_query_chain(llm, db)
response = chain.invoke({"question": "What are the three biggest hits?"})

# response is just our variable name. Here it contains the generated SQL.
# What did the model decide "biggest hits" means: sales, revenue, or something else?
display(Markdown(response))


To determine the "three biggest hits," I will assume it refers to tracks with the highest "UnitPrice" and use the "Track" table, sorting by "UnitPrice" in descending order and limiting the results to 3.

```sql
SQLQuery:
SELECT "Name", "UnitPrice" 
FROM "Track" 
ORDER BY "UnitPrice" DESC 
LIMIT 3;
```

In [41]:
# Inspect the prompt template chosen by LangChain for this SQL database.
# Look for the SQL instructions and the input/table_info placeholders.
# The chain fills those placeholders when invoked. This cell makes no model call.
chain.get_prompts()[0].pretty_print()


You are a SQLite expert. Given an input question, first create a syntactically correct SQLite query to run, then look at the results of the query and return the answer to the input question.
Unless the user specifies in the question a specific number of examples to obtain, query for at most 5 results using the LIMIT clause as per SQLite. You can order the results to return the most informative data in the database.
Never query for all columns from a table. You must query only the columns that are needed to answer the question. Wrap each column name in double quotes (") to denote them as delimited identifiers.
Pay attention to use only the column names you can see in the tables below. Be careful to not query for columns that do not exist. Also, pay attention to which column is in which table.
Pay attention to use date('now') function to get the current date, if the question involves "today".

Use the following format:

Question: Question here
SQLQuery: SQL Query to run
SQLResult: Result

### Your observations 4

What did the model take “the three biggest hits” to mean? Point to a column,
calculation, or ordering rule in its SQL. Then write a more precise version of
the question and identify where the prompt supplies the table information the
model needs. You only need to write your revised question here.

**Your response:**

The model took "the three biggest hits" to mean the tracks with the highest price. Its SQL selected Name and UnitPrice from the Track table, with ORDER BY UnitPrice DESC and LIMIT 3. The result was three items at 1.99, which tells me nothing about popularity. My more precise question is: Which three tracks have the most total units sold across all invoices? In the prompt, the table information is supplied in the table_info placeholder, right after the line "Only use the following tables". The chain fills that placeholder with the table definitions and sample rows, and my question goes into the input placeholder.

### M. Clean and execute the SQL

A model may wrap SQL in Markdown or add a label. `clean_sql_query` removes
common formatting so SQLite can read the query. It does not check whether
the query answers our question correctly.

Inspect `clean_response` before running it. `db.run` executes the SQL and
returns the result rows as text.

In [42]:
# re provides text-pattern matching. This helper handles response formatting;
# it does not verify SQL correctness or decide whether the query answers the question.
import re

def clean_sql_query(response):
    # Remove surrounding whitespace, then extract SQL from a code block if present.
    query = response.strip()
    fenced_sql = re.search(r"```(?:sql)?\s*(.*?)```", query, re.DOTALL | re.IGNORECASE)
    if fenced_sql:
        query = fenced_sql.group(1).strip()

    # The chain can stop before a closing code fence is generated.
    # Also remove a leading SQLQuery: label, which SQLite would not understand.
    query = re.sub(r"^```(?:sql)?\s*", "", query, flags=re.IGNORECASE)
    query = re.sub(r"^SQLQuery:\s*", "", query, flags=re.IGNORECASE)

    # Skip introductory prose and find a SELECT query or a WITH clause.
    # WITH names a temporary query result that the rest of the SQL can reference.
    sql_start = re.search(r"^\s*(?:SELECT|WITH)\b", query, re.MULTILINE | re.IGNORECASE)
    if sql_start is None:
        raise ValueError(f"No SELECT or WITH query found in model output: {response!r}")
    return query[sql_start.start():].strip()


In [43]:
# Clean the SQL we generated earlier and inspect it before running it.
# This is ordinary Python text processing, with no additional model call.
clean_response = clean_sql_query(response)
print(clean_response)


SELECT "Name", "UnitPrice" 
FROM "Track" 
ORDER BY "UnitPrice" DESC 
LIMIT 3;


In [44]:
# Now execute the query against SQLite. The database supplies the actual records.
# In this example, db.run returns a string showing the result rows as tuples.
# Check the query's joins, ordering, and limits before interpreting those rows.
db.run(clean_response)


"[('Battlestar Galactica: The Story So Far', 1.99), ('Occupation / Precipice', 1.99), ('Exodus, Pt. 1', 1.99)]"

### N. Connect SQL generation and execution

`QuerySQLDataBaseTool` lets us include database execution in a chain.
The sequence is **write SQL → clean SQL → execute SQL**. Its output is still
the database result, not a model-written explanation. For example, `[(8,)]`
is a result containing one row with the value 8.

The tool can return an SQL error as text, so read the output even when the
Python cell finishes without an exception.

In [45]:
from langchain_community.tools.sql_database.tool import QuerySQLDataBaseTool

# Wrap SQL execution as a LangChain tool so we can connect it to other steps.
# A tool here is called by our pipeline; the model is not choosing when to use it.
execute_query = QuerySQLDataBaseTool(db=db)
write_query = create_sql_query_chain(llm, db)

# The | operator passes each step's output to the next step:
# question dictionary -> SQL text -> cleaned SQL -> database result.
# We now assign chain to this larger pipeline, replacing the earlier SQL-only chain.
chain = write_query | clean_sql_query | execute_query

# This returns the query result, before any model-written explanation.
# The tool returns SQL errors as text too, so inspect the output for errors.
chain.invoke({"question": "How many employees are there"})


'[(8,)]'

### O. Turn the result into an answer

The answering prompt receives the original question, SQL, and database result.
`RunnablePassthrough.assign` keeps the existing dictionary fields while adding
`query` and then `result`. `itemgetter("query")` selects the SQL for execution.
Finally, the prompt and model turn the result into an answer.

In [46]:
from operator import itemgetter

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

# The answering model needs the original question, the SQL, and its result.
# The names inside braces are placeholders filled from a dictionary at runtime.
answer_prompt = PromptTemplate.from_template(
    """Given the following user question, corresponding SQL query, and SQL result, answer the user question.

Question: {question}
SQL Query: {query}
SQL Result: {result}
Answer: """
)

# Keep the original question as we add information to that dictionary:
# {question} -> {question, query} -> {question, query, result}.
# assign adds a field; itemgetter("query") selects just the SQL for execution.
chain = (
    RunnablePassthrough.assign(query=write_query | clean_sql_query)
    .assign(
        result=itemgetter("query") | execute_query
    )
    | answer_prompt      # Fill the template with the question, SQL, and result.
    | llm                # Ask the model to explain the database result.
    | StrOutputParser()  # Extract the model message's text as a Python string.
)

# Building this pipeline does not run it. Each invocation below makes two
# model calls: one to write SQL and one to turn the result into an answer.


In [47]:
# Run the full pipeline and compare its answer with the raw employee count above.
# response now holds the final answer, because chain now includes the answering step.
response = chain.invoke({"question": "How many employees are there?"})

display(Markdown(response))


There are 8 employees.

In [48]:
# The pipeline writes a fresh query each time; it does not reuse the earlier SQL.
# There is no conversation history here. This question is answered independently.
response = chain.invoke({"question": "What are the three biggest hits?"})

# Did the model use the same meaning of "biggest hits" as before?
# Try specifying "tracks ranked by total units sold across all invoices" instead.
display(Markdown(response))


The three biggest hits are:

1. **The Trooper** with 5 purchases.
2. **Untitled** with 4 purchases.
3. **The Number Of The Beast** with 4 purchases.

### Your observations 5

How many model calls does one invocation of this full chain make? Trace what
each call receives and produces, and identify the step performed by SQLite.
Use the employee-count example to explain why generating SQL is different from
retrieving the answer.

**Your response:**

One invocation of the full chain makes two model calls. The first call receives my question together with the table definitions and sample rows, and it produces SQL text. Then clean_sql_query removes the formatting, which is normal Python and not a model call. SQLite performs the step in the middle. It executes the SQL and returns the result, which was [(8,)] in the employee example. The second model call receives the question, the SQL and the result, and it produces the sentence "There are 8 employees." This is why generating SQL is different from retrieving the answer. The model never knew how many employees there are. It only wrote the instruction, and the number 8 came from the database. I also noticed that the full chain wrote new SQL for the biggest hits question. This time it counted purchases and answered The Trooper with 5 purchases, while the earlier SQL ordered by UnitPrice.

### P. Consider the limits of the evidence

The final examples ask for a promotion recommendation. Sales records do not
establish an employee's awareness of trends or whether they deserve a promotion.
Also consider what “last year” means for a database of historical records.

The model might generate SQL, decline, or produce text that the cleanup helper
cannot turn into SQL. If the full chain raises that error, record it and run
the following SQL-only cell separately. That cell makes a new request; it does
not reveal the exact query used in the earlier call.

In [49]:
# This question goes beyond what sales records can establish.
# A high sales count does not prove awareness of trends or merit a promotion.
# Also ask what date range "last year" should mean for this historical dataset.
response = chain.invoke({"question": "Which of our sales support agents should be promoted based on their awareness of trending genres, trending genres can be classified based on the last year of sales?"})

# Check whether the answer separates observed sales from unsupported conclusions.
display(Markdown(response))


The sales support agent who should be promoted based on their awareness of trending genres is **Margaret Park**. She achieved the highest sales in the trending genre over the last year, generating a total of **22.77** in sales.

In [50]:
# Return to a SQL-only chain to examine how a similar question becomes a query.
# This generates new SQL; it does not reveal the exact SQL used in the previous cell.
test_chain = create_sql_query_chain(llm, db)
response = test_chain.invoke({"question": "Which of our sales support agents should be promoted based on their awareness of trending genres, can be classified based on the last year of sales??"})

# Inspect the date filter, joins, and ranking rule. What can those actually measure?
# No SQL is executed by this cell.
display(Markdown(response))


To determine which sales support agent should be promoted based on their awareness of trending genres, we need to analyze the sales data for the last year. This involves identifying the most sold genre(s) and then associating those sales with the support agents who assisted customers making those purchases.

Here's the step-by-step plan:

1. Identify invoices from the last year using the `InvoiceDate`.
2. Join `Invoice` with `InvoiceLine` to get track sales and associate them with genres via the `Track` table.
3. Group the sales by genre and calculate the total sales per genre.
4. Identify the top genre(s) based on total sales.
5. Associate these sales with customers and their respective support agents from the `Customer` table.
6. Determine which sales support agent has the most influence in the trending genre(s).

SQLQuery:
```sql
WITH LastYearInvoices AS (
    SELECT "Invoice"."InvoiceId", "Invoice"."CustomerId"
    FROM "Invoice"
    WHERE DATE("InvoiceDate") >= DATE('now', '-1 year')
),
TrendingGenres AS (
    SELECT "Track"."GenreId", SUM("InvoiceLine"."Quantity") AS "TotalSales"
    FROM "LastYearInvoices"
    JOIN "InvoiceLine" ON "LastYearInvoices"."InvoiceId" = "InvoiceLine"."InvoiceId"
    JOIN "Track" ON "InvoiceLine"."TrackId" = "Track"."TrackId"
    GROUP BY "Track"."GenreId"
    ORDER BY "TotalSales" DESC
    LIMIT 1
),
AgentPerformance AS (
    SELECT "Employee"."FirstName", "Employee"."LastName", SUM("InvoiceLine"."Quantity") AS "AgentSales"
    FROM "LastYearInvoices"
    JOIN "Invoice" ON "LastYearInvoices"."InvoiceId" = "Invoice"."InvoiceId"
    JOIN "Customer" ON "Invoice"."CustomerId" = "Customer"."CustomerId"
    JOIN "Employee" ON "Customer"."SupportRepId" = "Employee"."EmployeeId"
    JOIN "InvoiceLine" ON "Invoice"."InvoiceId" = "InvoiceLine"."InvoiceId"
    JOIN "Track" ON "InvoiceLine"."TrackId" = "Track"."TrackId"
    WHERE "Track"."GenreId" = (SELECT "GenreId" FROM "TrendingGenres")
    GROUP BY "Employee"."EmployeeId"
    ORDER BY "AgentSales" DESC
    LIMIT 1
)
SELECT "FirstName", "LastName", "AgentSales"
FROM "AgentPerformance";
```

In [51]:
# My challenge question for Your observations 6.
my_question = "Which artist has the most tracks, and how many albums does that artist have?"

# SQL only, so I can inspect the generated query.
my_sql = test_chain.invoke({"question": my_question})
display(Markdown(my_sql))

# Full chain. This is a separate request.
my_answer = chain.invoke({"question": my_question})
display(Markdown(my_answer))

# My own check with a query I wrote myself.
check_query = """
SELECT ar.Name, COUNT(DISTINCT al.AlbumId) AS albums, COUNT(t.TrackId) AS tracks
FROM Artist ar
JOIN Album al ON al.ArtistId = ar.ArtistId
JOIN Track t ON t.AlbumId = al.AlbumId
GROUP BY ar.ArtistId
ORDER BY tracks DESC
LIMIT 3;
"""
display(pd.read_sql_query(check_query, engine))


```sql
SELECT 
    "Artist"."Name" AS "ArtistName", 
    COUNT(DISTINCT "Track"."TrackId") AS "TrackCount", 
    COUNT(DISTINCT "Album"."AlbumId") AS "AlbumCount"
FROM 
    "Artist"
JOIN 
    "Album" ON "Artist"."ArtistId" = "Album"."ArtistId"
JOIN 
    "Track" ON "Album"."AlbumId" = "Track"."AlbumId"
GROUP BY 
    "Artist"."ArtistId", "Artist"."Name"
ORDER BY 
    "TrackCount" DESC
LIMIT 1;
```

The artist with the most tracks is **Iron Maiden**, and they have **21 albums**.

,Name,albums,tracks
0,Iron Maiden,21,213
1,U2,10,135
2,Led Zeppelin,14,114


### Your observations 6

Try the challenge from class: ask a question that the database can answer but
that might lead the model to an incorrect answer. Choose a question involving
a comparison, a grouping, or more than one table; a question about someone's
private preferences would not test this.

Replace the question in an existing SQL-only cell to inspect the generated query,
then try it in an existing full-chain cell. These are separate requests and may
generate different SQL. In a few sentences, record your question, the answer,
and the evidence you used to check it. If you did not find a mistake, explain
what you checked rather than assuming a fluent answer is correct.

**Your response:**

My question was: Which artist has the most tracks, and how many albums does that artist have? I chose it because it needs the Artist, Album and Track tables. When Album is joined with Track, every album repeats one time for each track, so a simple count of albums would be wrong. The full chain answered Iron Maiden with 21 albums. In the SQL only cell, the generated query used COUNT(DISTINCT TrackId) and COUNT(DISTINCT AlbumId), grouped by artist and ordered by the track count, so it handled the part I was worried about. To check it, I wrote my own query in the same cell. It returned Iron Maiden with 21 albums and 213 tracks, and U2 was second with 135 tracks. So I did not find a mistake this time. However, the two cells are separate requests, so I could not see the exact SQL the full chain used, and I could only compare its final answer with my own query. The answer also left out the number of tracks, which my question did not ask for directly.

## Save your work

Save this notebook with your outputs and six responses. Keep the supplied data
files unchanged, then commit and push the notebook to your fork. Follow Canvas
for the submission link and deadline.